# Bootstrap da dissertação — V3

No Google Colab, execute as células na ordem e envie `dados_bootstrap.xlsx` quando solicitado. Em execução local, coloque essa planilha na mesma pasta do notebook.

## Atualização V3
A análise de sensibilidade compara 90%, 95%, 98% e 99% do patamar, mantendo fixas as medianas observadas e a definição do patamar. Compara a primeira vazão que supera cada velocidade interpolada com a primeira vazão que também atinge o patamar. Os resultados são exportados na aba `Sensibilidade` da planilha de saída.

O percentual de 95% é uma convenção operacional de aproximação ao patamar, não um limite físico universal nem uma determinação convencional de mínima fluidização. A análise complementar não calcula novos intervalos de confiança para os demais percentuais e não valida a superioridade de 95%.

## Procedimento preservado da V2
- U = Q/A, com diâmetro interno de 0,080 m, sem arredondamento intermediário. A coluna `U_ms` da planilha não é usada.
- 5.000 reamostragens, semente 2026 e 18 leituras por ponto.
- Ajuste nos sete primeiros pontos e patamar pela média das medianas dos dois últimos pontos.
- Figuras e estimativas calculadas pelo código.

Os intervalos representam a variabilidade propagada pela reamostragem, não toda a incerteza experimental. MAD não é calculado neste notebook.

Ao finalizar, salve uma cópia deste notebook com as saídas. A última seção registra as versões efetivamente utilizadas na execução.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from google.colab import files
except ImportError:
    files = None


In [ ]:
if files is not None:
    arquivo_enviado = files.upload()
else:
    caminho_entrada = Path("dados_bootstrap.xlsx")
    if not caminho_entrada.is_file():
        raise FileNotFoundError("Coloque dados_bootstrap.xlsx na pasta do notebook.")
    arquivo_enviado = {str(caminho_entrada): None}


In [ ]:
nome_arquivo = next(iter(arquivo_enviado))

df = pd.read_excel(nome_arquivo)

print("Dimensão da planilha:", df.shape)
display(df)

In [ ]:
Q = df["Q_Lmin"].to_numpy(dtype=float)
D_modelo = 0.080  # Diâmetro interno do modelo, em metros
A_modelo = np.pi * D_modelo**2 / 4

# Conversão de L/min para m³/s; sem arredondamento intermediário
U = (Q / 60000) / A_modelo

colunas_dp = [f"DP_{i:02d}" for i in range(1, 19)]
DP = df[colunas_dp].to_numpy(dtype=float)

print("Q:", Q)
print("U:", U)
print("Formato da matriz de ΔP:", DP.shape)

In [ ]:
# Verificação das condições de entrada do procedimento V2/V3.
if Q.shape != (9,) or DP.shape != (9, 18):
    raise ValueError("São necessários nove níveis de vazão e 18 leituras por nível.")
if not (np.isfinite(Q).all() and np.isfinite(DP).all()):
    raise ValueError("A planilha contém valores ausentes ou não finitos.")
if not np.allclose(Q, np.arange(60, 221, 20)):
    raise ValueError("Confira a ordem das vazões: 60, 80, ..., 220 L/min.")


In [ ]:
medianas_observadas = np.median(DP, axis=1)

verificacao = pd.DataFrame({
    "Q_Lmin": Q,
    "U_ms": U,
    "Mediana_DP": medianas_observadas
})

display(verificacao)

In [ ]:
def calcular_UdeltaP95(U, dp_mediana, n_pontos_patamar=2):
    """
    Calcula UdeltaP,95 como a velocidade na qual ΔP atinge
    95% do patamar médio dos últimos pontos.
    """

    patamar = np.mean(dp_mediana[-n_pontos_patamar:])
    dp_95 = 0.95 * patamar

    # Procura o primeiro intervalo que contém dp_95
    for i in range(len(U) - 1):
        y1 = dp_mediana[i]
        y2 = dp_mediana[i + 1]

        if (y1 <= dp_95 <= y2) or (y2 <= dp_95 <= y1):

            if y2 == y1:
                return U[i], patamar, dp_95

            UdeltaP95 = U[i] + (
                (dp_95 - y1) *
                (U[i + 1] - U[i]) /
                (y2 - y1)
            )

            return UdeltaP95, patamar, dp_95

    return np.nan, patamar, dp_95

In [ ]:
def calcular_umf_intersecao(
    U,
    dp_mediana,
    k_inicial=7,
    n_pontos_patamar=2
):
    """
    Ajusta uma reta aos k primeiros pontos e calcula
    sua interseção com o patamar médio dos últimos pontos.
    """

    U_inicial = U[:k_inicial]
    DP_inicial = dp_mediana[:k_inicial]

    coef_angular, intercepto = np.polyfit(
        U_inicial,
        DP_inicial,
        1
    )

    patamar = np.mean(dp_mediana[-n_pontos_patamar:])

    if coef_angular == 0:
        return np.nan, coef_angular, intercepto, patamar

    umf_int = (patamar - intercepto) / coef_angular

    return umf_int, coef_angular, intercepto, patamar

In [ ]:
UdeltaP95_pontual, patamar_pontual, dp95_pontual = calcular_UdeltaP95(
    U,
    medianas_observadas
)

umfint_pontual, a_pontual, b_pontual, patamar_int = (
    calcular_umf_intersecao(
        U,
        medianas_observadas
    )
)

print(f"Patamar: {patamar_pontual:.3f} mmH2O")
print(f"95% do patamar: {dp95_pontual:.3f} mmH2O")
print(f"UdeltaP,95: {UdeltaP95_pontual:.6f} m/s")
print(f"Umf,int: {umfint_pontual:.6f} m/s")
print(f"Reta inicial: ΔP = {a_pontual:.6f} U + {b_pontual:.6f}")

In [ ]:
B = 5000
n = DP.shape[1]

# Semente fixa para garantir reprodutibilidade computacional
rng = np.random.default_rng(2026)

vetor_UdeltaP95 = []
vetor_umfint = []

for b in range(B):

    medianas_bootstrap = np.empty(len(U))

    for i in range(len(U)):

        amostra_bootstrap = rng.choice(
            DP[i, :],
            size=n,
            replace=True
        )

        medianas_bootstrap[i] = np.median(
            amostra_bootstrap
        )

    UdeltaP95_b, _, _ = calcular_UdeltaP95(
        U,
        medianas_bootstrap,
        n_pontos_patamar=2
    )

    umfint_b, _, _, _ = calcular_umf_intersecao(
        U,
        medianas_bootstrap,
        k_inicial=7,
        n_pontos_patamar=2
    )

    if np.isfinite(UdeltaP95_b):
        vetor_UdeltaP95.append(UdeltaP95_b)

    if np.isfinite(umfint_b):
        vetor_umfint.append(umfint_b)

vetor_UdeltaP95 = np.asarray(vetor_UdeltaP95)
vetor_umfint = np.asarray(vetor_umfint)

print("Reamostragens válidas para UdeltaP,95:", len(vetor_UdeltaP95))
print("Reamostragens válidas para Umf,int:", len(vetor_umfint))

In [ ]:
ic_UdeltaP95 = np.percentile(
    vetor_UdeltaP95,
    [2.5, 97.5]
)

ic_umfint = np.percentile(
    vetor_umfint,
    [2.5, 97.5]
)

print("RESULTADOS BOOTSTRAP")
print("-" * 50)

print(
    f"UdeltaP,95 pontual = {UdeltaP95_pontual:.3f} m/s"
)

print(
    f"IC95% UdeltaP,95 = "
    f"{ic_UdeltaP95[0]:.3f} – "
    f"{ic_UdeltaP95[1]:.3f} m/s"
)

print()

print(
    f"Umf,int pontual = {umfint_pontual:.3f} m/s"
)

print(
    f"IC95% Umf,int = "
    f"{ic_umfint[0]:.3f} – "
    f"{ic_umfint[1]:.3f} m/s"
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    U,
    medianas_observadas,
    marker="o",
    label="Mediana experimental"
)

plt.axhline(
    patamar_pontual,
    linestyle="--",
    label=f"Patamar = {patamar_pontual:.1f} mmH₂O".replace(".", ",")
)

plt.axhline(
    dp95_pontual,
    linestyle=":",
    label=f"95% do patamar = {dp95_pontual:.1f} mmH₂O".replace(".", ",")
)

plt.axvline(
    UdeltaP95_pontual,
    linestyle="--",
    label=(r"$U_{\Delta P,95}$ = "
           + f"{UdeltaP95_pontual:.3f}".replace(".", ",") + " m/s")
)

plt.axvline(
    umfint_pontual,
    linestyle="-.",
    label=(r"$U_{\mathrm{mf,int}}$ = "
           + f"{umfint_pontual:.3f}".replace(".", ",") + " m/s")
)

plt.xlabel("Velocidade superficial, U (m/s)")
plt.ylabel("Queda de pressão, ΔP (mmH₂O)")
plt.title("Curva ΔP × U e velocidades características da transição")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "curva_DP_U_Umf.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    vetor_UdeltaP95,
    bins=30,
    edgecolor="black"
)

plt.axvline(
    ic_UdeltaP95[0],
    linestyle="--",
    label=f"P2,5% = {ic_UdeltaP95[0]:.3f}"
)

plt.axvline(
    ic_UdeltaP95[1],
    linestyle="--",
    label=f"P97,5% = {ic_UdeltaP95[1]:.3f}"
)

plt.xlabel("UdeltaP,95 (m/s)")
plt.ylabel("Frequência")
plt.title("Distribuição bootstrap de UdeltaP,95")
plt.legend()
plt.tight_layout()

plt.savefig(
    "bootstrap_UdeltaP95.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    vetor_umfint,
    bins=30,
    edgecolor="black"
)

plt.axvline(
    ic_umfint[0],
    linestyle="--",
    label=f"P2,5% = {ic_umfint[0]:.3f}"
)

plt.axvline(
    ic_umfint[1],
    linestyle="--",
    label=f"P97,5% = {ic_umfint[1]:.3f}"
)

plt.xlabel("Umf,int (m/s)")
plt.ylabel("Frequência")
plt.title("Distribuição bootstrap de Umf,int")
plt.legend()
plt.tight_layout()

plt.savefig(
    "bootstrap_Umfint.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Sensibilidade ao percentual do patamar

Esta comparação usa a curva observada, sem reamostragem. A estabilidade da seleção pelo critério completo decorre principalmente da exigência de atingir o patamar. Ela não constitui validação independente do limiar de 95%.


In [ ]:
# Sensibilidade determinística. Não altera as variáveis do bootstrap.
percentuais_sens = [90, 95, 98, 99]
patamar_sens = float(patamar_pontual)
if not np.allclose(medianas_observadas[-2:], 120.0):
    raise ValueError("As medianas de 200 e 220 L/min diferem do patamar documentado de 120 mmH2O.")
if not np.all(np.diff(medianas_observadas) >= 0):
    raise ValueError("A curva observada não é crescente: revisar a regra de primeiro cruzamento.")
resultados_sens = []
for percentual_sens in percentuais_sens:
    alvo_sens = patamar_sens * percentual_sens / 100.0
    indices_sens = np.flatnonzero(medianas_observadas >= alvo_sens)
    if len(indices_sens) == 0 or indices_sens[0] == 0:
        raise ValueError("Limiar fora da faixa de interpolação disponível.")
    j_sens = int(indices_sens[0])
    dp_inf_sens = medianas_observadas[j_sens - 1]
    dp_sup_sens = medianas_observadas[j_sens]
    u_limiar_sens = U[j_sens - 1] + (
        (alvo_sens - dp_inf_sens) / (dp_sup_sens - dp_inf_sens)
        * (U[j_sens] - U[j_sens - 1])
    )
    mask_vel_sens = U > u_limiar_sens
    mask_completo_sens = mask_vel_sens & (
        (medianas_observadas > patamar_sens)
        | np.isclose(medianas_observadas, patamar_sens)
    )
    if not mask_vel_sens.any() or not mask_completo_sens.any():
        raise ValueError("Nenhum nível ensaiado atende ao critério avaliado.")
    resultados_sens.append({
        "Percentual (%)": percentual_sens,
        "DeltaP de referência (mmH2O)": alvo_sens,
        "U interpolada (m/s)": float(u_limiar_sens),
        "Primeira Q apenas por U (L/min)": float(Q[mask_vel_sens][0]),
        "Primeira Q pelo critério completo (L/min)": float(Q[mask_completo_sens][0]),
    })
tabela_sens = pd.DataFrame(resultados_sens)
assert np.isclose(tabela_sens.loc[1, "U interpolada (m/s)"], UdeltaP95_pontual)
print(tabela_sens.to_string(index=False, float_format=lambda v: f"{v:.6f}"))


In [ ]:
resultado_resumo = pd.DataFrame({
    "Estimador": ["UdeltaP,95", "Umf,int"],
    "Estimativa_pontual_m_s": [
        UdeltaP95_pontual,
        umfint_pontual
    ],
    "IC95_inferior_m_s": [
        ic_UdeltaP95[0],
        ic_umfint[0]
    ],
    "IC95_superior_m_s": [
        ic_UdeltaP95[1],
        ic_umfint[1]
    ],
    "B": [B, B],
    "Trecho_inicial_k": [
        "Não aplicável",
        7
    ],
    "Pontos_patamar": [2, 2]
})

distribuicoes = pd.DataFrame({
    "UdeltaP95_bootstrap": pd.Series(vetor_UdeltaP95),
    "Umfint_bootstrap": pd.Series(vetor_umfint)
})

with pd.ExcelWriter(
    "resultados_bootstrap_Umf_V3.xlsx",
    engine="openpyxl"
) as writer:

    resultado_resumo.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    verificacao.to_excel(
        writer,
        sheet_name="Medianas",
        index=False
    )

    tabela_sens.to_excel(writer, sheet_name="Sensibilidade", index=False)

    distribuicoes.to_excel(
        writer,
        sheet_name="Distribuicoes",
        index=False
    )

display(resultado_resumo)

In [ ]:
if files is not None:
    files.download("resultados_bootstrap_Umf_V3.xlsx")
else:
    print("Arquivos salvos na pasta de execução do notebook.")


In [ ]:
if files is not None:
    files.download("curva_DP_U_Umf.png")
    files.download("bootstrap_UdeltaP95.png")
    files.download("bootstrap_Umfint.png")
else:
    print("Arquivos salvos na pasta de execução do notebook.")


In [ ]:
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib
from datetime import datetime

print("Python:", platform.python_version())
print("Implementação:", platform.python_implementation())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Data da execução:", datetime.now().strftime("%d/%m/%Y %H:%M"))
print("Descrição completa do Python:", sys.version)

In [ ]:
import openpyxl
print("openpyxl:", openpyxl.__version__)